In [1]:
from pathlib import Path
import os

PROJECT_ROOT_BOOT = Path.cwd().resolve()
if PROJECT_ROOT_BOOT.name.lower() == 'notebooks':
    PROJECT_ROOT_BOOT = PROJECT_ROOT_BOOT.parent
HF_DATASETS_CACHE_DIR = PROJECT_ROOT_BOOT / 'data' / '.hf_datasets_cache'
HF_DATASETS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_DATASETS_CACHE'] = str(HF_DATASETS_CACHE_DIR)
print('Hugging Face 数据集缓存:', HF_DATASETS_CACHE_DIR)


Hugging Face 数据集缓存: D:\dev\projects\fourlang_translation\data\.hf_datasets_cache


In [2]:
from pathlib import Path
import gc
import importlib
import importlib.metadata as metadata
import json
import os
import random
import re
import sys
import time
from collections import Counter

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
HF_COMPAT_DIR = PROJECT_ROOT / '.hf_compat'
if HF_COMPAT_DIR.exists():
    compat_path = str(HF_COMPAT_DIR)
    if compat_path in sys.path:
        sys.path.remove(compat_path)
    sys.path.insert(0, compat_path)
    importlib.invalidate_caches()

import numpy as np
import pandas as pd
import sacrebleu
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

TRAIN_FILE = PROJECT_ROOT / 'data' / 'clean' / 'en_uz' / 'public_5k_v2' / 'train_directed_v2.jsonl'
VALID_FILE = PROJECT_ROOT / 'data' / 'clean' / 'en_uz' / 'public_5k_v2' / 'validation_directed_v2.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'lora' / 'm2m100_en_uz_public_v2'
RESULT_DIR = PROJECT_ROOT / 'results' / 'm2m100_en_uz_public_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_BASE_MODEL = Path(r'C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636')
BASE_MODEL = str(LOCAL_BASE_MODEL) if (LOCAL_BASE_MODEL / 'pytorch_model.bin').exists() else 'facebook/m2m100_418M'
FALLBACK_CACHE_DIR = PROJECT_ROOT / 'models' / 'huggingface' / 'hub'

SEED = 42
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
MAX_STEPS = 900
EVAL_STEPS = 150
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 16

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.set_num_threads(4)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('基础模型:', BASE_MODEL)
print('训练数据:', TRAIN_FILE)
print('输出目录:', OUTPUT_DIR)
print('最大训练步数:', MAX_STEPS, '有效批量:', GRADIENT_ACCUMULATION_STEPS)
print('当前 Cell 不会开始训练。')


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


基础模型: C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M\snapshots\55c2e61bbf05dfb8d7abccdc3fae6fc8512fd636
训练数据: D:\dev\projects\fourlang_translation\data\clean\en_uz\public_5k_v2\train_directed_v2.jsonl
输出目录: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_public_v2
最大训练步数: 900 有效批量: 16
当前 Cell 不会开始训练。


In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), 'Device:', DEVICE, 'BF16:', USE_BF16, 'FP16:', USE_FP16)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print(f'显存: {props.total_memory / 1024**3:.2f} GB')
assert torch.cuda.is_available(), '未检测到 CUDA GPU，请先确认 Notebook 使用项目虚拟环境内核。'


PyTorch: 2.13.0+cu132
CUDA: True Device: cuda BF16: True FP16: False
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
显存: 7.96 GB


In [4]:
for path in (TRAIN_FILE, VALID_FILE):
    assert path.exists(), f'缺少数据文件: {path}'
train_dataset = load_dataset('json', data_files=str(TRAIN_FILE), split='train')
valid_dataset = load_dataset('json', data_files=str(VALID_FILE), split='train')

REQUIRED = {'pair_id', 'src_lang', 'tgt_lang', 'src_text', 'tgt_text'}
for name, dataset in [('train', train_dataset), ('validation', valid_dataset)]:
    missing = REQUIRED.difference(dataset.column_names)
    assert not missing, f'{name} 缺少字段: {sorted(missing)}'
    assert not any(not str(row['src_text']).strip() or not str(row['tgt_text']).strip() for row in dataset), f'{name} 存在空文本'
    directions = Counter(f"{row['src_lang']}-{row['tgt_lang']}" for row in dataset)
    print(name, len(dataset), directions, '句对:', len(set(dataset['pair_id'])))

train_pair_ids = set(train_dataset['pair_id'])
valid_pair_ids = set(valid_dataset['pair_id'])
assert train_pair_ids.isdisjoint(valid_pair_ids), '训练和验证存在句对泄漏'
assert Counter(f"{row['src_lang']}-{row['tgt_lang']}" for row in train_dataset)['en-uz'] == Counter(f"{row['src_lang']}-{row['tgt_lang']}" for row in train_dataset)['uz-en']
print('训练/验证无 pair_id 泄漏，双向数量平衡。')
display(pd.DataFrame(train_dataset.select(range(8))))


train 9500 Counter({'en-uz': 4750, 'uz-en': 4750}) 句对: 4750
validation 500 Counter({'en-uz': 250, 'uz-en': 250}) 句对: 250
训练/验证无 pair_id 泄漏，双向数量平衡。


,pair_id,source,source_version,license,commercial_status,labse_score,src_lang,tgt_lang,src_text,tgt_text
0,4bd1974b437710acd86b,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.986685,en,uz,Other statistics,Boshqa statistikalari
1,4bd1974b437710acd86b,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.986685,uz,en,Boshqa statistikalari,Other statistics
2,2f8b64694a2c0c133dcc,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.986515,en,uz,Other names,Boshqa nomlari
3,2f8b64694a2c0c133dcc,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.986515,uz,en,Boshqa nomlari,Other names
4,7b5cd56b259b2d87a981,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.984223,en,uz,Other places,Boshqa joylar
5,7b5cd56b259b2d87a981,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.984223,uz,en,Boshqa joylar,Other places
6,3a3f7df557bc47c8c35a,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.983687,en,uz,He reigned from 1722 to 1735.,U 1722-yildan 1735-yilgacha hukmronlik qildi.
7,3a3f7df557bc47c8c35a,wikimedia,v20260327,Upstream Wikimedia terms vary; commonly CC BY-...,attribution_sharealike_legal_review_required,0.983687,uz,en,U 1722-yildan 1735-yilgacha hukmronlik qildi.,He reigned from 1722 to 1735.


In [5]:
gc.collect()
torch.cuda.empty_cache()
load_kwargs = {'low_cpu_mem_usage': True}
if BASE_MODEL == 'facebook/m2m100_418M':
    load_kwargs['cache_dir'] = str(FALLBACK_CACHE_DIR)
    print('本地快照不存在，将下载到:', FALLBACK_CACHE_DIR)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, cache_dir=str(FALLBACK_CACHE_DIR) if BASE_MODEL == 'facebook/m2m100_418M' else None)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, **load_kwargs)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.config.use_cache = False
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'可训练比例: {trainable / total:.4%}')
assert trainable / total < 0.02, '基础模型可能没有正确冻结'


trainable params: 2,359,296 || all params: 486,264,832 || trainable%: 0.4852
可训练比例: 0.4852%


In [6]:
def preprocess(example):
    tokenizer.src_lang = example['src_lang']
    model_inputs = tokenizer(example['src_text'], max_length=MAX_SOURCE_LENGTH, truncation=True)
    tokenizer.src_lang = example['tgt_lang']
    labels = tokenizer(example['tgt_text'], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_dataset.map(preprocess, remove_columns=train_dataset.column_names, desc='Tokenize train')
tokenized_valid = valid_dataset.map(preprocess, remove_columns=valid_dataset.column_names, desc='Tokenize validation')
print('INPUT:', tokenizer.decode(tokenized_train[0]['input_ids'], skip_special_tokens=False))
print('LABEL:', tokenizer.decode(tokenized_train[0]['labels'], skip_special_tokens=False))
print('最长输入限制:', MAX_SOURCE_LENGTH, '最长输出限制:', MAX_TARGET_LENGTH)


Tokenize validation: 100%|██████████| 500/500 [00:00<00:00, 4371.81 examples/s]

INPUT: __en__ Other statistics</s>
LABEL: __uz__ Boshqa statistikalari</s>
最长输入限制: 96 最长输出限制: 96


In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    max_grad_norm=1.0,
    bf16=USE_BF16,
    fp16=USE_FP16,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    logging_steps=10,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    predict_with_generate=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    save_safetensors=True,
)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)
trainer = Seq2SeqTrainer(
    model=model, args=training_args, train_dataset=tokenized_train, eval_dataset=tokenized_valid,
    tokenizer=tokenizer, data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print('Trainer 已创建，但尚未训练。')


C:\Users\WingYouther\AppData\Local\Temp\ipykernel_52852\3390975893.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
max_steps is given, it will override any value given in num_train_epochs


Trainer 已创建，但尚未训练。


In [8]:
model = model.to(DEVICE)
model.train()
torch.cuda.reset_peak_memory_stats()
batch = collator([tokenized_train[0]])
batch = {key: value.to(DEVICE) for key, value in batch.items()}
with torch.autocast(device_type='cuda', dtype=torch.bfloat16 if USE_BF16 else torch.float16):
    outputs = model(**batch)
    loss = outputs.loss
assert torch.isfinite(loss), f'Loss 异常: {loss.item()}'
assert loss.requires_grad, 'Loss 没有梯度'
trainer.accelerator.backward(loss)
model.zero_grad(set_to_none=True)
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f'训练预检 Loss: {loss.item():.4f}')
print(f'显存峰值: {peak_gb:.2f} GB')
assert peak_gb < 7.7, '显存峰值过高，请将长度从96降到80。'
del batch, outputs, loss
gc.collect()
torch.cuda.empty_cache()
print('显存预检通过。')


训练预检 Loss: 7.2489
显存峰值: 2.13 GB
显存预检通过。


In [9]:
@torch.inference_mode()
def translate_one(text, src_lang, tgt_lang, model_obj=model):
    model_obj.eval()
    model_obj.config.use_cache = True
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LENGTH).to(DEVICE)
    generated = model_obj.generate(
        **inputs, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=MAX_TARGET_LENGTH, num_beams=1,
    )
    model_obj.config.use_cache = False
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

for src_lang, tgt_lang, text in [
    ('en', 'uz', 'I will go to the airport tomorrow morning.'),
    ('uz', 'en', 'Men ertaga ertalab aeroportga boraman.'),
]:
    print(src_lang, '->', tgt_lang, '|', translate_one(text, src_lang, tgt_lang))
model.train()


D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


en -> uz | Yarın sabah aerodinamikga qaytarga.
uz -> en | It is also the airport.


PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): M2M100ForConditionalGeneration(
      (model): M2M100Model(
        (shared): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
        (encoder): M2M100Encoder(
          (embed_tokens): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
          (embed_positions): M2M100SinusoidalPositionalEmbedding()
          (layers): ModuleList(
            (0-11): 12 x M2M100EncoderLayer(
              (self_attn): M2M100SdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=16, bias=False)
                  )
                 

In [13]:
RUN_TRAINING = True
RESUME_FROM_CHECKPOINT = False

if RUN_TRAINING:
    model.config.use_cache = False
    train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    final_adapter_dir = OUTPUT_DIR / 'final_adapter'
    trainer.save_model(str(final_adapter_dir))
    tokenizer.save_pretrained(final_adapter_dir)
    trainer.save_metrics('train', train_result.metrics)
    print('训练完成，最佳 Adapter 已保存:', final_adapter_dir)
else:
    print('尚未训练。确认前面所有 Cell 正常后，把 RUN_TRAINING 改为 True。')
    print('中断后续跑时，保持 RUN_TRAINING=True，并把 RESUME_FROM_CHECKPOINT 改为 True。')


Step,Training Loss,Validation Loss
150,3.264200,2.899837
300,3.027200,2.719946
450,2.830900,2.630880
600,2.769900,2.575340
750,2.703700,2.543445
900,2.748300,2.533944


训练完成，最佳 Adapter 已保存: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_public_v2\final_adapter


In [15]:
from datasets import load_dataset,tqdm
RUN_FINAL_VALIDATION = True
FINAL_ADAPTER_DIR = OUTPUT_DIR / 'final_adapter'

def evaluate_generation(model_obj, dataset, name):
    model_obj.eval()
    model_obj.config.use_cache = True
    rows = []
    for item in tqdm(dataset, total=len(dataset), desc=name):
        started = time.perf_counter()
        prediction = translate_one(item['src_text'], item['src_lang'], item['tgt_lang'], model_obj)
        rows.append({
            'pair_id': item['pair_id'], 'direction': f"{item['src_lang']}-{item['tgt_lang']}",
            'source': item['src_text'], 'reference': item['tgt_text'], 'prediction': prediction,
            'latency_seconds': time.perf_counter() - started,
        })
    frame = pd.DataFrame(rows)
    metrics = {}
    for direction, group in frame.groupby('direction'):
        preds, refs = group['prediction'].tolist(), group['reference'].tolist()
        metrics[direction] = {
            'samples': len(group),
            'bleu': sacrebleu.corpus_bleu(preds, [refs]).score,
            'chrf2': sacrebleu.corpus_chrf(preds, [refs], word_order=2).score,
            'latency_mean_seconds': group['latency_seconds'].mean(),
            'latency_p95_seconds': group['latency_seconds'].quantile(.95),
        }
    frame.to_csv(RESULT_DIR / f'predictions_{name}.csv', index=False, encoding='utf-8-sig')
    (RESULT_DIR / f'metrics_{name}.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    return frame, metrics

if RUN_FINAL_VALIDATION:
    if not FINAL_ADAPTER_DIR.exists():
        print('还没有 final_adapter，请先完成训练。')
    else:
        model.eval()
        validation_predictions, validation_metrics = evaluate_generation(model, valid_dataset, 'public_v2_adapter')
        print(json.dumps(validation_metrics, ensure_ascii=False, indent=2))
        display(validation_predictions.sample(min(20, len(validation_predictions)), random_state=SEED))
else:
    print('已跳过最终生成评测。')


public_v2_adapter:   0%|          | 0/500 [00:00<?, ?it/s]D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
public_v2_adapter: 100%|██████████| 500/500 [02:55<00:00,  2.84it/s]

{
  "en-uz": {
    "samples": 250,
    "bleu": 17.588781470852542,
    "chrf2": 32.19262105883877,
    "latency_mean_seconds": 0.4421766039961949,
    "latency_p95_seconds": 0.9583176249754615
  },
  "uz-en": {
    "samples": 250,
    "bleu": 25.379052907196186,
    "chrf2": 46.92919520535226,
    "latency_mean_seconds": 0.2595834131967276,
    "latency_p95_seconds": 0.49238794504199174
  }
}


,pair_id,direction,source,reference,prediction,latency_seconds
361,b6e0f90124a8ed22f157,uz-en,U Holmenkollen medalini qo'lga kiritgan birinc...,She is the first Swedish woman to win the Holm...,She was the first winner to win the Holmenkoll...,0.164558
73,a854d728f8ddcfc46566,uz-en,Qishloq Helsinkidan taxminan 40 km uzoqlikda j...,The village is approximately 40 km from Helsin...,The city is about 40 km from Helsinki and abou...,0.169621
374,2a9ee7a4f4dfe6fad1e6,en-uz,"In October 1997, he released his second album ...","1997-yil oktyabr oyida u o'zining ikkinchi ""En...",1997-yil oktobda o'z o'z o'z o'z o'z o'z o'z o...,0.911989
155,085283caee044b5a0c84,uz-en,"U Monnou daryosi bo'yida, taxminan 2 miles (3....","It is located beside the River Monnow, about 2...","It is located near Monnou, 2 miles (3.2 km) fr...",0.410165
104,35189ce241c97a97615e,en-uz,She served as a United States magistrate judge...,U 2018-yildan 2022-yilgacha xuddi shu sudning ...,U 2018-yildan 2022-yilda o'ynagan o'ynagan o'y...,0.460805
394,88854dce4d88600f9ad2,en-uz,Kurtzuba married Joshua Coakley in 2005.,Kurtzuba 2005-yilda Joshua Coakleyga turmushga...,Kurtzuba 2005-yilda Joshua Coakleyga o'zilgan.,0.188318
377,b1e0c0a18432c70e3cff,uz-en,1958-yil uchun Chevrolet modellari 1957-yildag...,"For 1958, Chevrolet models were redesigned lon...","By 1958, Chevrolet models were developed in 19...",0.257739
124,322c95bee6f7cd86881a,en-uz,"Since 1983, she has served as an ESPN analyst ...","1983-yildan boshlab u. turli tadbirlar, jumlad...","1983-yildan bu yana, NCAA basketbol ma'yashida...",0.428965
68,1bba27b459d95d3c243d,en-uz,It is used mostly for football matches and is ...,U asosan futbol o'yinlari uchun ishlatiladi va...,U u'zilgan futbol ma'yashlarga o'zilgan va Eks...,0.388050
450,c6075c48f652625ccf64,en-uz,"In December 2005, Little Rimando married her l...",2005-yil dekabr oyida Little Rimando o'zining ...,2005-yilda 2005-yilda Little Rimando o'zilgan ...,0.308663


In [16]:
if FINAL_ADAPTER_DIR.exists():
    del model
    gc.collect()
    torch.cuda.empty_cache()
    reload_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    reload_base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, low_cpu_mem_usage=True)
    reload_model = PeftModel.from_pretrained(reload_base, FINAL_ADAPTER_DIR).to(DEVICE).eval()
    reload_model.config.use_cache = True

    def translate_reloaded(text, src_lang, tgt_lang):
        reload_tokenizer.src_lang = src_lang
        inputs = reload_tokenizer(text, return_tensors='pt', truncation=True, max_length=96).to(DEVICE)
        with torch.inference_mode():
            generated = reload_model.generate(
                **inputs, forced_bos_token_id=reload_tokenizer.get_lang_id(tgt_lang),
                max_new_tokens=96, num_beams=1,
            )
        return reload_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

    print('EN->UZ:', translate_reloaded('I will go to the airport tomorrow.', 'en', 'uz'))
    print('UZ->EN:', translate_reloaded('Men ertaga aeroportga boraman.', 'uz', 'en'))
    print('Adapter 重载成功:', FINAL_ADAPTER_DIR)
else:
    print('训练完成后再运行本 Cell 验证 Adapter 重载。')


D:\dev\projects\fourlang_translation\.hf_compat\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


EN->UZ: U hamga u aeroportda qaytarga.
UZ->EN: But I still stay at the airport.
Adapter 重载成功: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_public_v2\final_adapter
